In [10]:

"""
===========================================================
build_features.py

Build per-user-per-day features for CERT R4.2
Ground truth labels come ONLY from insiders.csv

Author: Team Member 2
===========================================================
"""

import os
import json
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------

import pandas as pd

logon = pd.read_csv("logon.csv")
device = pd.read_csv("device.csv")
file_data = pd.read_csv("file.csv")
insiders = pd.read_csv("insiders.csv")

print("Datasets Loaded Successfully!\n")

print("Logon :", logon.shape)
print("Device:", device.shape)
print("File  :", file_data.shape)
print("Insiders:", insiders.shape)
OUTPUT_DIR = "ml/data"

RANDOM_STATE = 42
NORMAL_SAMPLE_USERS = 300

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("CERT R4.2 FEATURE BUILD PIPELINE")
print("=" * 60)

# ----------------------------------------------------------
# LOAD DATA
# ----------------------------------------------------------

print("\nLoading datasets...")


print("Logon :", logon.shape)
print("Device:", device.shape)
print("File  :", file_data.shape)
print("Insiders:", insiders.shape)

# ----------------------------------------------------------
# KEEP ONLY CERT R4.2 INSIDERS
# ----------------------------------------------------------

insiders = insiders[insiders["dataset"] == 4.2].copy()

print("\nCERT R4.2 Insider Users:", insiders["user"].nunique())

# ----------------------------------------------------------
# DATE CONVERSION
# ----------------------------------------------------------

print("\nParsing timestamps...")

logon["date"] = pd.to_datetime(
    logon["date"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

device["date"] = pd.to_datetime(
    device["date"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

file_data["date"] = pd.to_datetime(
    file_data["date"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

insiders["start"] = pd.to_datetime(
    insiders["start"],
    errors="coerce"
)

insiders["end"] = pd.to_datetime(
    insiders["end"],
    errors="coerce"
)

print("Timestamp conversion completed.")

# ----------------------------------------------------------
# REMOVE BAD ROWS
# ----------------------------------------------------------

logon = logon.dropna(subset=["date", "user"])
device = device.dropna(subset=["date", "user"])
file_data = file_data.dropna(subset=["date", "user"])

# ----------------------------------------------------------
# EXTRACT DATE + HOUR
# ----------------------------------------------------------

for df in [logon, device, file_data]:

    df["day"] = df["date"].dt.date

    df["hour"] = df["date"].dt.hour

# ----------------------------------------------------------
# OFF HOURS
# ----------------------------------------------------------

def off_hour(hour):

    return int(hour >= 20 or hour < 6)

logon["off_hour"] = logon["hour"].apply(off_hour)
device["off_hour"] = device["hour"].apply(off_hour)
file_data["off_hour"] = file_data["hour"].apply(off_hour)

# ----------------------------------------------------------
# KEEP ONLY REQUIRED EVENTS
# ----------------------------------------------------------

logon = logon[
    logon["activity"].isin(["Logon", "Logoff"])
]

device = device[
    device["activity"].isin(["Connect", "Disconnect"])
]

print("\nData Ready")

print("Logon Rows :", len(logon))
print("Device Rows:", len(device))
print("File Rows  :", len(file_data))

print("\nDate Range")

print(logon["date"].min(), "->", logon["date"].max())


# ==========================================================
# BUILD DAILY LOGIN FEATURES
# ==========================================================

print("\nBuilding Login Features...")

login_daily = (
    logon.groupby(["user", "day"])
    .agg(
        total_logins=("activity", "count"),
        unique_login_pcs=("pc", "nunique"),
        avg_login_hour=("hour", "mean"),
        off_hour_logins=("off_hour", "sum"),
    )
    .reset_index()
)

print("Login feature rows :", len(login_daily))


# ==========================================================
# BUILD DAILY DEVICE FEATURES
# ==========================================================

print("\nBuilding Device Features...")

device_daily = (
    device.groupby(["user", "day"])
    .agg(
        total_device_events=("activity", "count"),
        unique_devices=("pc", "nunique"),
        off_hour_device=("off_hour", "sum"),
    )
    .reset_index()
)

print("Device feature rows :", len(device_daily))


# ==========================================================
# BUILD DAILY FILE FEATURES
# ==========================================================

print("\nBuilding File Features...")

file_daily = (
    file_data.groupby(["user", "day"])
    .agg(
        files_accessed=("filename", "count"),
        unique_files=("filename", "nunique"),
        unique_content=("content", "nunique"),
        off_hour_file=("off_hour", "sum"),
    )
    .reset_index()
)

print("File feature rows :", len(file_daily))


# ==========================================================
# MERGE ALL DAILY FEATURES
# ==========================================================

print("\nMerging Daily Features...")

features = login_daily.merge(
    device_daily,
    on=["user", "day"],
    how="outer",
)

features = features.merge(
    file_daily,
    on=["user", "day"],
    how="outer",
)

print("Merged rows :", len(features))


# ==========================================================
# REPLACE MISSING VALUES
# ==========================================================

numeric_columns = [

    "total_logins",

    "unique_login_pcs",

    "avg_login_hour",

    "off_hour_logins",

    "total_device_events",

    "unique_devices",

    "off_hour_device",

    "files_accessed",

    "unique_files",

    "unique_content",

    "off_hour_file",

]

for column in numeric_columns:

    features[column] = features[column].fillna(0)


# ==========================================================
# DERIVED FEATURES
# ==========================================================

print("\nCreating Derived Features...")


features["files_per_login"] = np.where(

    features["total_logins"] > 0,

    features["files_accessed"] / features["total_logins"],

    0,

)


features["device_login_ratio"] = np.where(

    features["total_logins"] > 0,

    features["total_device_events"] / features["total_logins"],

    0,

)


offhour_total = (

    features["off_hour_logins"]

    + features["off_hour_device"]

    + features["off_hour_file"]

)


activity_total = (

    features["total_logins"]

    + features["total_device_events"]

    + features["files_accessed"]

)


features["off_hour_ratio"] = np.where(

    activity_total > 0,

    offhour_total / activity_total,

    0,

)


print("\nDerived Features Completed")


# ==========================================================
# FEATURE LIST
# ==========================================================

feature_columns = [

    "total_logins",

    "unique_login_pcs",

    "avg_login_hour",

    "off_hour_logins",

    "total_device_events",

    "unique_devices",

    "off_hour_device",

    "files_accessed",

    "unique_files",

    "unique_content",

    "off_hour_file",

    "files_per_login",

    "device_login_ratio",

    "off_hour_ratio",

]

print("\nNumber of Features :", len(feature_columns))

print(feature_columns)

# ==========================================================
# CREATE GROUND TRUTH LABELS FROM insiders.csv
# ==========================================================

print("\nCreating Ground Truth Labels...")

# Create label column
features["risk_label"] = 0

# Convert day to datetime
features["day"] = pd.to_datetime(features["day"])

# Keep only valid insider rows
insiders = insiders.dropna(subset=["user", "start", "end"])

# Filter only CERT R4.2
insiders = insiders[insiders["dataset"] == 4.2].copy()

# Add ±1 day padding
insiders["label_start"] = insiders["start"] - pd.Timedelta(days=1)
insiders["label_end"] = insiders["end"] + pd.Timedelta(days=1)

print("R4.2 Insider Users :", insiders["user"].nunique())

# ----------------------------------------------------------
# LABEL USER-DAYS
# ----------------------------------------------------------

for _, insider in insiders.iterrows():

    user = insider["user"]

    start = insider["label_start"].normalize()

    end = insider["label_end"].normalize()

    mask = (

        (features["user"] == user)

        &

        (features["day"] >= start)

        &

        (features["day"] <= end)

    )

    features.loc[mask, "risk_label"] = 1

print("Ground Truth Labels Assigned")


# ==========================================================
# CLASS BALANCE
# ==========================================================

print("\nClass Distribution")

class_counts = features["risk_label"].value_counts().sort_index()

print(class_counts)

positive = int(features["risk_label"].sum())

negative = int(len(features) - positive)

print("\nNormal User-Days :", negative)
print("Insider User-Days:", positive)

print(
    "Positive Ratio : {:.4f}%".format(
        100 * positive / len(features)
    )
)


# ==========================================================
# VERIFY INSIDER USERS
# ==========================================================

print("\nVerifying Labels...")

labelled_users = features.loc[
    features["risk_label"] == 1,
    "user"
].unique()

print("Users labelled malicious :", len(labelled_users))

missing_users = set(insiders["user"]) - set(labelled_users)

if len(missing_users) == 0:
    print("All insider users successfully labelled.")
else:
    print("WARNING")
    print("These users were not found:")
    print(sorted(list(missing_users)))


# ==========================================================
# FEATURE PREVIEW
# ==========================================================

print("\nFeature Preview")

print(features.head())

print("\nFeature Shape")

print(features.shape)


# ==========================================================
# SAMPLE USERS
# Keep all insider users + 300 normal users
# ==========================================================

print("\nSampling Users...")

# All insider users
insider_users = set(insiders["user"].unique())

# All users in features
all_users = set(features["user"].unique())

# Normal users
normal_users = list(all_users - insider_users)

print("Total Users           :", len(all_users))
print("Insider Users         :", len(insider_users))
print("Available Normal Users:", len(normal_users))

# Sample normal users
sample_size = min(NORMAL_SAMPLE_USERS, len(normal_users))

sampled_normal_users = random.sample(
    normal_users,
    sample_size
)

selected_users = insider_users.union(sampled_normal_users)

features = features[
    features["user"].isin(selected_users)
].copy()

print("\nUsers after sampling :", features["user"].nunique())


# ==========================================================
# SORT DATA
# ==========================================================

features = features.sort_values(
    ["user", "day"]
).reset_index(drop=True)


# ==========================================================
# FINAL VALIDATION
# ==========================================================

print("\nFinal Dataset Summary")

print("Rows :", len(features))
print("Users:", features["user"].nunique())

print("\nRisk Label Distribution")

print(features["risk_label"].value_counts())

print("\nMissing Values")

print(features.isnull().sum())


# ==========================================================
# SAVE FEATURE NAMES
# ==========================================================

feature_json = os.path.join(
    OUTPUT_DIR,
    "feature_names.json"
)

with open(feature_json, "w") as f:

    json.dump(feature_columns, f, indent=4)

print("\nFeature names saved:")
print(feature_json)


# ==========================================================
# SAVE PARQUET
# ==========================================================

parquet_file = os.path.join(
    OUTPUT_DIR,
    "features_daily.parquet"
)

features.to_parquet(
    parquet_file,
    index=False
)

print("\nFeature dataset saved:")
print(parquet_file)


# ==========================================================
# OPTIONAL CSV EXPORT
# ==========================================================

csv_file = os.path.join(
    OUTPUT_DIR,
    "features_daily.csv"
)

features.to_csv(
    csv_file,
    index=False
)

print(csv_file)


# ==========================================================
# PREVIEW
# ==========================================================

print("\nPreview")

print(features.head())


print("\nColumns")

print(features.columns.tolist())


print("\nDataset Shape")

print(features.shape)


print("\nFinished Successfully!")

print("=" * 60)

Datasets Loaded Successfully!

Logon : (854859, 5)
Device: (405380, 5)
File  : (406833, 6)
Insiders: (191, 6)
CERT R4.2 FEATURE BUILD PIPELINE

Loading datasets...
Logon : (854859, 5)
Device: (405380, 5)
File  : (406833, 6)
Insiders: (191, 6)

CERT R4.2 Insider Users: 70

Parsing timestamps...
Timestamp conversion completed.

Data Ready
Logon Rows : 854859
Device Rows: 405380
File Rows  : 406833

Date Range
2010-01-02 06:49:00 -> 2011-05-17 06:43:35

Building Login Features...
Login feature rows : 330452

Building Device Features...
Device feature rows : 55717

Building File Features...
File feature rows : 42133

Merging Daily Features...
Merged rows : 330452

Creating Derived Features...

Derived Features Completed

Number of Features : 14
['total_logins', 'unique_login_pcs', 'avg_login_hour', 'off_hour_logins', 'total_device_events', 'unique_devices', 'off_hour_device', 'files_accessed', 'unique_files', 'unique_content', 'off_hour_file', 'files_per_login', 'device_login_ratio', 'off_

In [5]:
!python ml/pipelines/build_features.py

python3: can't open file '/content/ml/pipelines/build_features.py': [Errno 2] No such file or directory


In [7]:
%%writefile build_features.py

Overwriting build_features.py


In [11]:
!python build_features.py

In [12]:
!ls

build_features.py  file.csv	 logon.csv  sample_data
device.csv	   insiders.csv  ml


In [13]:
import os
print(os.path.exists("build_features.py"))

True


In [14]:
!ls ml/data

feature_names.json  features_daily.csv	features_daily.parquet


In [15]:
%%writefile train.py

Writing train.py


In [16]:
!python train.py

In [17]:
!ls ml/models

ls: cannot access 'ml/models': No such file or directory


In [18]:
import pandas as pd

df = pd.read_parquet("ml/data/features_daily.parquet")

print(df.columns.tolist())

['user', 'day', 'total_logins', 'unique_login_pcs', 'avg_login_hour', 'off_hour_logins', 'total_device_events', 'unique_devices', 'off_hour_device', 'files_accessed', 'unique_files', 'unique_content', 'off_hour_file', 'files_per_login', 'device_login_ratio', 'off_hour_ratio', 'risk_label']


In [19]:
print(df.head())

      user        day  total_logins  unique_login_pcs  avg_login_hour  \
0  AAE0190 2010-01-04             2                 1            13.0   
1  AAE0190 2010-01-05             2                 1            13.0   
2  AAE0190 2010-01-06             2                 1            13.0   
3  AAE0190 2010-01-07             2                 1            13.0   
4  AAE0190 2010-01-08             2                 1            13.0   

   off_hour_logins  total_device_events  unique_devices  off_hour_device  \
0                0                  0.0             0.0              0.0   
1                0                  0.0             0.0              0.0   
2                0                  0.0             0.0              0.0   
3                0                  0.0             0.0              0.0   
4                0                  0.0             0.0              0.0   

   files_accessed  unique_files  unique_content  off_hour_file  \
0             0.0           0.0       

In [20]:
import pandas as pd

df = pd.read_parquet("ml/data/features_daily.parquet")

print(df[["total_device_events", "files_accessed"]].describe())

print("\nRows with device activity:",
      (df["total_device_events"] > 0).sum())

print("Rows with file activity:",
      (df["files_accessed"] > 0).sum())

print("Positive labels:",
      df["risk_label"].sum())

       total_device_events  files_accessed
count        115797.000000   115797.000000
mean              1.347686        1.314870
std               3.825448        4.689033
min               0.000000        0.000000
25%               0.000000        0.000000
50%               0.000000        0.000000
75%               0.000000        0.000000
max              28.000000       54.000000

Rows with device activity: 22513
Rows with file activity: 16831
Positive labels: 1460


In [33]:
%%writefile train.py
"""
===========================================================
train.py

Train Random Forest and Isolation Forest
using daily CERT features

Author: Team Member 2
===========================================================
"""

import os
import json
import joblib
import warnings

import numpy as np
import pandas as pd

import sklearn

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import IsolationForest

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

print("=" * 60)
print("CERT R4.2 TRAINING PIPELINE")
print("=" * 60)

print("\nScikit-Learn Version")
print(sklearn.__version__)

# =====================================================
# PATHS
# =====================================================

FEATURE_FILE = "ml/data/features_daily.parquet"

MODEL_DIR = "ml/models"

os.makedirs(MODEL_DIR, exist_ok=True)

# =====================================================
# LOAD FEATURES
# =====================================================

print("\nLoading Features...")

df = pd.read_parquet(FEATURE_FILE)

print(df.shape)

print(df.head())

# =====================================================
# FEATURE LIST
# =====================================================

feature_columns = [

    "total_logins",

    "unique_login_pcs",

    "avg_login_hour",

    "off_hour_logins",

    "total_device_events",

    "unique_devices",

    "off_hour_device",

    "files_accessed",

    "unique_files",

    "unique_content",

    "off_hour_file",

    "files_per_login",

    "device_login_ratio",

    "off_hour_ratio",

]

X = df[feature_columns]

y = df["risk_label"]

groups = df["user"]

print("\nNumber of Features :", len(feature_columns))

print("Users :", groups.nunique())

print("Positive Samples :", y.sum())

print("Negative Samples :", len(y) - y.sum())

# =====================================================
# USER LEVEL SPLIT
# =====================================================

print("\nCreating User-Level Split...")

# =====================================================
# IDENTIFY USERS CORRECTLY
# =====================================================

# Users with at least one malicious day
insider_users = set(
    df.loc[df["risk_label"] == 1, "user"].unique()
)

# All users
all_users = set(df["user"].unique())

# Users that are never insiders
normal_users = all_users - insider_users

# Convert back to arrays
insider_users = np.array(sorted(list(insider_users)))
normal_users = np.array(sorted(list(normal_users)))

print("\nUser Summary")
print("Total Users   :", len(all_users))
print("Insider Users :", len(insider_users))
print("Normal Users  :", len(normal_users))
# Split insiders

train_insiders, test_insiders = train_test_split(

    insider_users,

    test_size=0.20,

    random_state=42

)

# Split normals

train_normals, test_normals = train_test_split(

    normal_users,

    test_size=0.20,

    random_state=42

)

train_users = set(train_insiders).union(train_normals)

test_users = set(test_insiders).union(test_normals)

train_df = df[df["user"].isin(train_users)].copy()

test_df = df[df["user"].isin(test_users)].copy()

print("\nTraining Rows :", len(train_df))

print("Testing Rows :", len(test_df))

print("Training Users :", train_df["user"].nunique())

print("Testing Users :", test_df["user"].nunique())

# =====================================================
# TRAIN / TEST MATRICES
# =====================================================

X_train = train_df[feature_columns]

X_test = test_df[feature_columns]

y_train = train_df["risk_label"]

y_test = test_df["risk_label"]

print("\nTrain Shape :", X_train.shape)

print("Test Shape :", X_test.shape)

print("\nPart 1 Completed Successfully.")

# =====================================================
# PART 2 - ISOLATION FOREST
# =====================================================

print("\n" + "=" * 60)
print("TRAINING ISOLATION FOREST")
print("=" * 60)

# -----------------------------------------------------
# Train ONLY on normal training rows
# -----------------------------------------------------

normal_train = train_df[train_df["risk_label"] == 0]

X_train_normal = normal_train[feature_columns]

print("Normal Training Samples :", len(X_train_normal))

# -----------------------------------------------------
# Build Model
# -----------------------------------------------------

iso_model = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

print("\nTraining Isolation Forest...")

iso_model.fit(X_train_normal)

print("Isolation Forest Training Completed!")

# -----------------------------------------------------
# Generate anomaly scores
# -----------------------------------------------------

print("\nGenerating anomaly scores...")

train_df["if_score"] = -iso_model.decision_function(X_train)

test_df["if_score"] = -iso_model.decision_function(X_test)

# Higher score = More anomalous

train_auc = roc_auc_score(
    train_df["risk_label"],
    train_df["if_score"]
)

test_auc = roc_auc_score(
    test_df["risk_label"],
    test_df["if_score"]
)

print("\nIsolation Forest ROC-AUC")
print(f"Train ROC-AUC : {train_auc:.4f}")
print(f"Test  ROC-AUC : {test_auc:.4f}")

# -----------------------------------------------------
# Save Model
# -----------------------------------------------------

iso_model_path = os.path.join(
    MODEL_DIR,
    "isolation_forest_v2.pkl"
)

joblib.dump(
    iso_model,
    iso_model_path
)

print("\nIsolation Forest Saved Successfully!")

print(iso_model_path)

print("\nPart 2 Completed Successfully!")

# =====================================================
# PART 3 - RANDOM FOREST
# =====================================================

print("\n" + "=" * 60)
print("TRAINING RANDOM FOREST")
print("=" * 60)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\nTraining Random Forest...")

rf_model.fit(X_train, y_train)

print("Random Forest Training Completed!")

# =====================================================
# PREDICTIONS
# =====================================================

print("\nGenerating Predictions...")

train_pred = rf_model.predict(X_train)
test_pred = rf_model.predict(X_test)

train_prob = rf_model.predict_proba(X_train)[:, 1]
test_prob = rf_model.predict_proba(X_test)[:, 1]

# =====================================================
# METRICS
# =====================================================

print("\nRandom Forest Performance")

train_auc = roc_auc_score(y_train, train_prob)
test_auc = roc_auc_score(y_test, test_prob)

print(f"Train ROC-AUC : {train_auc:.4f}")
print(f"Test  ROC-AUC : {test_auc:.4f}")

print("\nClassification Report")

print(classification_report(
    y_test,
    test_pred,
    digits=4
))

print("\nConfusion Matrix")

cm = confusion_matrix(
    y_test,
    test_pred
)

print(cm)

# =====================================================
# SAVE MODEL
# =====================================================

rf_model_path = os.path.join(
    MODEL_DIR,
    "random_forest_v2.pkl"
)

joblib.dump(
    rf_model,
    rf_model_path
)

print("\nRandom Forest Saved!")

print(rf_model_path)

# =====================================================
# SAVE FEATURE NAMES
# =====================================================

feature_file = os.path.join(
    MODEL_DIR,
    "feature_names.json"
)

with open(feature_file, "w") as f:

    json.dump(feature_columns, f, indent=4)

print("\nFeature Names Saved!")

print(feature_file)

print("\nPart 3 Completed Successfully!")

# =====================================================
# PART 4 - SAVE METRICS & FINAL OUTPUTS
# =====================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("\n" + "="*60)
print("FINAL MODEL EVALUATION")
print("="*60)

# -----------------------------------------------------
# RANDOM FOREST METRICS
# -----------------------------------------------------

accuracy = accuracy_score(y_test, test_pred)
precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, test_prob)

print("\nRandom Forest Results")
print("--------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

# -----------------------------------------------------
# ISOLATION FOREST METRICS
# -----------------------------------------------------

iso_auc = roc_auc_score(
    test_df["risk_label"],
    test_df["if_score"]
)

print("\nIsolation Forest")
print("--------------------------")
print(f"ROC-AUC : {iso_auc:.4f}")

# -----------------------------------------------------
# SAVE METRICS
# -----------------------------------------------------

metrics = {

    "random_forest": {

        "accuracy": float(accuracy),

        "precision": float(precision),

        "recall": float(recall),

        "f1_score": float(f1),

        "roc_auc": float(roc_auc)

    },

    "isolation_forest": {

        "roc_auc": float(iso_auc)

    },

    "dataset": {

        "train_rows": int(len(train_df)),

        "test_rows": int(len(test_df)),

        "features": feature_columns

    }

}

metrics_path = os.path.join(
    MODEL_DIR,
    "metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print("\nmetrics.json Saved!")

# -----------------------------------------------------
# SAVE FEATURE NAMES
# -----------------------------------------------------

feature_path = os.path.join(
    MODEL_DIR,
    "feature_names.json"
)

with open(feature_path, "w") as f:
    json.dump(feature_columns, f, indent=4)

print("feature_names.json Saved!")

# -----------------------------------------------------
# VERIFY FILES
# -----------------------------------------------------

print("\nSaved Files")
print("--------------------------")

for file in os.listdir(MODEL_DIR):
    print(file)

print("\nTraining Pipeline Completed Successfully!")
print("="*60)

Overwriting train.py


In [34]:
!python train.py

CERT R4.2 TRAINING PIPELINE

Scikit-Learn Version
1.6.1

Loading Features...
(115797, 17)
      user        day  ...  off_hour_ratio  risk_label
0  AAE0190 2010-01-04  ...             0.0           0
1  AAE0190 2010-01-05  ...             0.0           0
2  AAE0190 2010-01-06  ...             0.0           0
3  AAE0190 2010-01-07  ...             0.0           0
4  AAE0190 2010-01-08  ...             0.0           0

[5 rows x 17 columns]

Number of Features : 14
Users : 370
Positive Samples : 1460
Negative Samples : 114337

Creating User-Level Split...

User Summary
Total Users   : 370
Insider Users : 70
Normal Users  : 300

Training Rows : 93581
Testing Rows : 22216
Training Users : 296
Testing Users : 74

Train Shape : (93581, 14)
Test Shape : (22216, 14)

Part 1 Completed Successfully.

TRAINING ISOLATION FOREST
Normal Training Samples : 92409

Training Isolation Forest...
Isolation Forest Training Completed!

Generating anomaly scores...

Isolation Forest ROC-AUC
Train ROC-AUC : 0

In [36]:
!ls ml/models

feature_names.json  isolation_forest_v2.pkl  metrics.json  random_forest_v2.pkl


In [37]:
import json

with open("ml/models/metrics.json") as f:
    metrics = json.load(f)

print(metrics)

{'random_forest': {'accuracy': 0.916411595246669, 'precision': 0.10833749375936096, 'recall': 0.7534722222222222, 'f1_score': 0.18943692710606722, 'roc_auc': 0.8942525601463375}, 'isolation_forest': {'roc_auc': 0.7777305905184646}, 'dataset': {'train_rows': 93581, 'test_rows': 22216, 'features': ['total_logins', 'unique_login_pcs', 'avg_login_hour', 'off_hour_logins', 'total_device_events', 'unique_devices', 'off_hour_device', 'files_accessed', 'unique_files', 'unique_content', 'off_hour_file', 'files_per_login', 'device_login_ratio', 'off_hour_ratio']}}


In [69]:
!rm evaluate.py

In [84]:
%%writefile evaluate.py

# =====================================================
# CERT R4.2 EVALUATION PIPELINE (FINAL VERSION)
# =====================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score
)

import joblib


print("="*60)
print("CERT R4.2 EVALUATION")
print("="*60)


print("\nScikit-Learn Version")
print(sklearn.__version__)


# =====================================================
# CREATE OUTPUT DIRECTORIES
# =====================================================

os.makedirs(
    "docs/paper/figures",
    exist_ok=True
)


# =====================================================
# LOAD DATA
# =====================================================

print("\nLoading Features...")


df = pd.read_parquet(
    "ml/data/features_daily.parquet"
)


print(df.shape)

print(df.head())


# =====================================================
# LOAD MODELS
# =====================================================

rf_model = joblib.load(
    "ml/models/random_forest_v2.pkl"
)


iso_model = joblib.load(
    "ml/models/isolation_forest_v2.pkl"
)


print("\nModels Loaded Successfully!")


# =====================================================
# FEATURES
# =====================================================


with open(
    "ml/models/feature_names.json",
    "r"
) as f:

    feature_columns = json.load(f)


print("\nFeatures:",len(feature_columns))


X = df[feature_columns]

y = df["risk_label"]


print("Positive Samples:",sum(y))
print("Negative Samples:",len(y)-sum(y))


# =====================================================
# RECREATE TEST SPLIT
# SAME USER LEVEL SPLIT AS TRAIN
# =====================================================

from sklearn.model_selection import GroupShuffleSplit


groups = df["user"]


splitter = GroupShuffleSplit(
    test_size=0.2,
    random_state=42
)


train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups
    )
)


X_test = X.iloc[test_idx]

y_test = y.iloc[test_idx]


test_df = df.iloc[test_idx].copy()


print("\nTest Shape:")
print(X_test.shape)



# =====================================================
# MODEL PREDICTIONS
# =====================================================

print("\nGenerating Predictions...")


rf_prob = rf_model.predict_proba(
    X_test
)[:,1]


rf_pred = (
    rf_prob >= 0.5
).astype(int)



# Isolation Forest

iso_decision = iso_model.decision_function(
    X_test
)


iso_pred = iso_model.predict(
    X_test
)


# convert:
# normal = 1
# anomaly = -1

iso_pred = (
    iso_pred == -1
).astype(int)



print("Predictions Generated Successfully!")


print("\nRandom Forest ROC-AUC:",
      round(
          roc_auc_score(
              y_test,
              rf_prob
          ),
          4
      )
)


print("\nPART 1 COMPLETED")


# =====================================================
# PART 2 - MODEL EVALUATION METRICS
# =====================================================

print("\n" + "="*60)
print("MODEL METRICS")
print("="*60)


# -----------------------------------------------------
# Random Forest Metrics
# -----------------------------------------------------

print("\nRandom Forest Classification Report\n")

report = classification_report(
    y_test,
    rf_pred,
    digits=4
)

print(report)


rf_auc = roc_auc_score(
    y_test,
    rf_prob
)


print(
    "Random Forest ROC-AUC:",
    round(rf_auc,4)
)


# -----------------------------------------------------
# Confusion Matrix
# -----------------------------------------------------

cm = confusion_matrix(
    y_test,
    rf_pred
)


plt.figure(
    figsize=(6,5)
)


plt.imshow(cm)

plt.title(
    "Random Forest Confusion Matrix"
)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Actual"
)


for i in range(2):
    for j in range(2):

        plt.text(
            j,
            i,
            cm[i,j],
            ha="center",
            va="center"
        )


plt.colorbar()


plt.savefig(
    "docs/paper/figures/confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close()


print("Confusion Matrix Saved!")


# -----------------------------------------------------
# ROC Curve
# -----------------------------------------------------

fpr, tpr, _ = roc_curve(
    y_test,
    rf_prob
)


plt.figure(
    figsize=(7,6)
)


plt.plot(
    fpr,
    tpr,
    label=f"Random Forest AUC={rf_auc:.3f}"
)


plt.plot(
    [0,1],
    [0,1],
    linestyle="--"
)


plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve"
)


plt.legend()


plt.savefig(
    "docs/paper/figures/roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close()


print("ROC Curve Saved!")


print("\nPART 2 COMPLETED")


# =====================================================
# PART 3 - SHAP ANALYSIS
# =====================================================

import shap


print("\n" + "="*60)
print("SHAP ANALYSIS")
print("="*60)


# -----------------------------------------------------
# Create SHAP Explainer
# -----------------------------------------------------

print("Creating TreeExplainer...")

explainer = shap.TreeExplainer(
    rf_model
)


# Use 1000 samples only

X_sample = X_test.sample(
    min(1000, len(X_test)),
    random_state=42
)


print("Calculating SHAP values...")


shap_values = explainer(
    X_sample
)


print(type(shap_values))

print(
    "Shape :",
    shap_values.values.shape
)



# -----------------------------------------------------
# Select Insider Class (Class 1)
# -----------------------------------------------------

shap_class1 = shap.Explanation(

    values=shap_values.values[:,:,1],

    base_values=shap_values.base_values[:,1],

    data=X_sample.values,

    feature_names=feature_columns

)



# -----------------------------------------------------
# SHAP Beeswarm
# -----------------------------------------------------

print("Generating Beeswarm...")


plt.figure(
    figsize=(10,6)
)


shap.plots.beeswarm(
    shap_class1,
    max_display=14,
    show=False
)


plt.savefig(
    "docs/paper/figures/shap_beeswarm.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close()


print("Beeswarm Saved!")



# -----------------------------------------------------
# TOP 10 SHAP FEATURES
# -----------------------------------------------------

mean_shap = np.abs(
    shap_class1.values
).mean(axis=0)


shap_importance = pd.DataFrame({

    "Feature":feature_columns,

    "MeanSHAP":mean_shap

})


shap_importance = shap_importance.sort_values(
    "MeanSHAP",
    ascending=False
)



top10_shap = shap_importance.head(10).copy()


top10_shap["Weight(%)"] = (

    top10_shap["MeanSHAP"]

    /

    top10_shap["MeanSHAP"].sum()

)*100



print("\nTop 10 Features")

print(top10_shap)



top10_shap.to_csv(

    "docs/paper/figures/top10_shap.csv",

    index=False

)


print("Top10 SHAP CSV Saved!")



# -----------------------------------------------------
# DATASET SUMMARY
# -----------------------------------------------------

summary = pd.DataFrame({

    "Metric":[

        "Total User Days",

        "Test User Days",

        "Total Users",

        "Positive Labels",

        "Negative Labels"

    ],


    "Value":[

        len(df),

        len(test_df),

        df["user"].nunique(),

        int(df["risk_label"].sum()),

        int((df["risk_label"]==0).sum())

    ]

})



print("\nDataset Summary")

print(summary)



summary.to_csv(

    "docs/paper/dataset_summary.csv",

    index=False

)


print("\nPART 3 COMPLETED")


# =====================================================
# PART 4 - ISOLATION FOREST EVALUATION
# =====================================================

print("\n" + "="*60)
print("ISOLATION FOREST EVALUATION")
print("="*60)


# -----------------------------------------------------
# Convert Isolation Forest scores
# -----------------------------------------------------

# Isolation Forest:
# lower decision_function = more anomalous

anomaly_score = -iso_model.decision_function(
    X_test
)


# -----------------------------------------------------
# ROC-AUC
# -----------------------------------------------------

iso_auc = roc_auc_score(
    y_test,
    anomaly_score
)


print(
    "Isolation Forest ROC-AUC:",
    round(iso_auc,4)
)



# -----------------------------------------------------
# Precision@K and Recall@K
# -----------------------------------------------------

k_values = [
    50,
    100,
    200
]


results = []


# Create ranking dataframe

rank_df = pd.DataFrame({

    "anomaly_score": anomaly_score,

    "true_label": y_test.values

})


rank_df = rank_df.sort_values(
    by="anomaly_score",
    ascending=False
)



total_positive = rank_df["true_label"].sum()



for k in k_values:


    top_k = rank_df.head(k)


    precision_k = (
        top_k["true_label"].sum()
        /
        k
    )


    recall_k = (

        top_k["true_label"].sum()

        /

        total_positive

    )


    results.append({

        "K":k,

        "Precision@K":precision_k,

        "Recall@K":recall_k

    })



iso_results = pd.DataFrame(results)



print("\nPrecision@K and Recall@K")

print(iso_results)



# Save table

iso_results.to_csv(

    "docs/paper/figures/isolation_forest_precision_recall.csv",

    index=False

)



# -----------------------------------------------------
# Append results
# -----------------------------------------------------

with open(
    "docs/paper/results-template.md",
    "a"
) as f:


    f.write(
        "\n\n## Isolation Forest Results\n\n"
    )


    f.write(
        iso_results.to_markdown(index=False)
    )


print("\nIsolation Forest Results Saved!")


print("\nPART 4 COMPLETED")



# =====================================================
# PART 5 - TRUST SCORE ANALYSIS
# =====================================================

print("\n" + "="*60)
print("TRUST SCORE ANALYSIS")
print("="*60)


# -----------------------------------------------------
# Convert Isolation Forest decision function
# -----------------------------------------------------

anomaly01 = 1 / (
    1 + np.exp(
        10 * iso_decision
    )
)


# -----------------------------------------------------
# Trust Score Formula
# -----------------------------------------------------

identity_confidence = 0.5


trust_score = 100 * (

    0.4 * (1 - anomaly01)

    +

    0.4 * (1 - rf_prob)

    +

    0.2 * identity_confidence

)



trust_df = pd.DataFrame({

    "trust_score": trust_score,

    "label": y_test.values

})


print(
    trust_df.head()
)



# -----------------------------------------------------
# Trust Score ROC-AUC
# -----------------------------------------------------

trust_auc = roc_auc_score(

    y_test,

    -trust_score

)


print(

    "Trust Score ROC-AUC:",

    round(trust_auc,4)

)



# -----------------------------------------------------
# Distribution Plot
# -----------------------------------------------------

plt.figure(
    figsize=(8,6)
)


plt.hist(

    trust_df[
        trust_df["label"]==0
    ]["trust_score"],

    bins=30,

    alpha=0.6,

    label="Normal"

)


plt.hist(

    trust_df[
        trust_df["label"]==1
    ]["trust_score"],

    bins=30,

    alpha=0.6,

    label="Insider"

)



plt.xlabel(
    "Trust Score"
)


plt.ylabel(
    "Count"
)


plt.title(
    "Trust Score Distribution"
)


plt.legend()


plt.savefig(

    "docs/paper/figures/trust_score_distribution.png",

    dpi=300,

    bbox_inches="tight"

)


plt.close()


print(
    "Trust Score Distribution Saved!"
)



# -----------------------------------------------------
# Save Results
# -----------------------------------------------------

trust_results = pd.DataFrame({

    "Metric":[

        "Trust Score ROC-AUC",

        "Identity Confidence",

        "RF Weight",

        "IF Weight"

    ],

    "Value":[

        trust_auc,

        0.5,

        0.4,

        0.4

    ]

})



trust_results.to_csv(

    "docs/paper/figures/trust_score_results.csv",

    index=False

)



with open(

    "docs/paper/results-template.md",

    "a"

) as f:


    f.write(

        "\n\n## Trust Score Results\n\n"

    )


    f.write(

        trust_results.to_markdown(index=False)

    )



print(
    "\nTrust Score Results Saved!"
)


print(
    "\nPART 5 COMPLETED"
)


# =====================================================
# PART 6 - TRUST SCORE WEIGHT SENSITIVITY
# =====================================================

print("\n" + "="*60)
print("TRUST SCORE WEIGHT SENSITIVITY")
print("="*60)


weight_configs = [

    (0.4,0.4,0.2),

    (0.5,0.3,0.2),

    (0.3,0.5,0.2),

    (0.5,0.5,0.0),

    (0.33,0.33,0.34)

]


sensitivity_results = []



for rf_w, iso_w, id_w in weight_configs:


    # Trust score

    score = 100 * (

        rf_w * (1-rf_prob)

        +

        iso_w * (1-anomaly01)

        +

        id_w * identity_confidence

    )


    # ROC-AUC
    auc = roc_auc_score(

        y_test,

        -score

    )


    restrict_percent = (

        (score < 40).sum()

        /

        len(score)

    ) * 100



    mfa_percent = (

        (score < 70).sum()

        /

        len(score)

    ) * 100



    sensitivity_results.append({

        "RF Weight":rf_w,

        "IF Weight":iso_w,

        "Identity Weight":id_w,

        "Trust ROC-AUC":auc,

        "Below 40 (%)":restrict_percent,

        "Below 70 (%)":mfa_percent

    })



sensitivity_df = pd.DataFrame(
    sensitivity_results
)



print("\nWeight Sensitivity Results")

print(sensitivity_df)



# Save CSV

sensitivity_df.to_csv(

    "docs/paper/figures/trust_weight_sensitivity.csv",

    index=False

)



# Append to paper results

with open(

    "docs/paper/results-template.md",

    "a"

) as f:


    f.write(

        "\n\n## Trust Score Weight Sensitivity\n\n"

    )


    f.write(

        sensitivity_df.to_markdown(index=False)

    )



print(
    "\nWeight Sensitivity Saved!"
)


print(
    "\nPART 6 COMPLETED"
)


# =====================================================
# PART 7 - OFF-HOURS ABLATION
# =====================================================

print("\n" + "="*60)
print("OFF-HOURS ABLATION")
print("="*60)


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score


# -----------------------------------------------------
# Remove off-hour features
# -----------------------------------------------------

offhour_features = [

    "off_hour_ratio",

    "off_hour_logins",

    "off_hour_device",

    "off_hour_file"

]


ablation_features = [

    f for f in feature_columns

    if f not in offhour_features

]


print(
    "Original Features:",
    len(feature_columns)
)


print(
    "Ablation Features:",
    len(ablation_features)
)



# -----------------------------------------------------
# Prepare data
# -----------------------------------------------------

X_train_ab = df.iloc[train_idx][ablation_features]

y_train_ab = df.iloc[train_idx]["risk_label"]


X_test_ab = df.iloc[test_idx][ablation_features]

y_test_ab = df.iloc[test_idx]["risk_label"]



# -----------------------------------------------------
# Train Ablation RF
# -----------------------------------------------------

print("\nTraining Ablation Random Forest...")


rf_ablation = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    min_samples_split=5,

    min_samples_leaf=2,

    class_weight="balanced",

    random_state=42

)



rf_ablation.fit(

    X_train_ab,

    y_train_ab

)



print(
    "Ablation Training Completed!"
)



# -----------------------------------------------------
# Predictions
# -----------------------------------------------------

ab_prob = rf_ablation.predict_proba(

    X_test_ab

)[:,1]


ab_pred = (

    ab_prob >= 0.5

).astype(int)



# -----------------------------------------------------
# Metrics
# -----------------------------------------------------

full_f1 = f1_score(

    y_test,

    rf_pred,

    average="macro"

)


ab_f1 = f1_score(

    y_test_ab,

    ab_pred,

    average="macro"

)



full_auc = roc_auc_score(

    y_test,

    rf_prob

)


ab_auc = roc_auc_score(

    y_test_ab,

    ab_prob

)



ablation_results = pd.DataFrame({

    "Model":[

        "Full Random Forest",

        "No Off-Hours Features"

    ],


    "F1_macro":[

        full_f1,

        ab_f1

    ],


    "ROC_AUC":[

        full_auc,

        ab_auc

    ],


    "F1_Delta":[

        0,

        ab_f1-full_f1

    ],


    "AUC_Delta":[

        0,

        ab_auc-full_auc

    ]

})


print("\nAblation Results")

print(ablation_results)



# -----------------------------------------------------
# Save model
# -----------------------------------------------------

os.makedirs(

    "ml/models",

    exist_ok=True

)



joblib.dump(

    rf_ablation,

    "ml/models/rf_ablation_no_offhours.pkl"

)



print(

    "\nAblation Model Saved!"

)



# Save results

ablation_results.to_csv(

    "docs/paper/figures/offhours_ablation.csv",

    index=False

)



with open(

    "docs/paper/results-template.md",

    "a"

) as f:


    f.write(

        "\n\n## Off-hours Ablation Results\n\n"

    )


    f.write(

        ablation_results.to_markdown(index=False)

    )



print(

    "\nPART 7 COMPLETED"

)

Overwriting evaluate.py


In [85]:
!python evaluate.py

CERT R4.2 EVALUATION

Scikit-Learn Version
1.6.1

Loading Features...
(115797, 17)
      user        day  ...  off_hour_ratio  risk_label
0  AAE0190 2010-01-04  ...             0.0           0
1  AAE0190 2010-01-05  ...             0.0           0
2  AAE0190 2010-01-06  ...             0.0           0
3  AAE0190 2010-01-07  ...             0.0           0
4  AAE0190 2010-01-08  ...             0.0           0

[5 rows x 17 columns]

Models Loaded Successfully!

Features: 14
Positive Samples: 1460
Negative Samples: 114337

Test Shape:
(23318, 14)

Generating Predictions...
Predictions Generated Successfully!

Random Forest ROC-AUC: 0.9181

PART 1 COMPLETED

MODEL METRICS

Random Forest Classification Report

              precision    recall  f1-score   support

           0     0.9984    0.9454    0.9712     23106
           1     0.1237    0.8396    0.2156       212

    accuracy                         0.9445     23318
   macro avg     0.5611    0.8925    0.5934     23318
weighted av

In [75]:
!pip install -q shap